# Finetuning RoBERTa for Same-Corpus Inference

This notebook finetunes a RoBERTa where training and inference are in the same data. This is a large model, therefore, we won't be able to run all 10 runs in the same session.

## Set up Libraries

In [ ]:
!pip install transformers datasets evaluate

# Load Data

RoBERTa will be trained from the original text data in `Data` folder

In [ ]:
dataset = 'SQUAD_GPT4'

import pandas as pd
import numpy as np

data = pd.read_csv(f'Data/{dataset}.csv', encoding = "ISO-8859-1")
data.head()

In [ ]:
seed = 1111
np.random.seed(seed)

sampling_indices = np.random.uniform(0,1,data.shape[0])
train_data = data.loc[sampling_indices < 0.9]
test_data = data.loc[sampling_indices >= 0.9]

train_data = pd.DataFrame({'text':pd.concat([train_data['answer'], train_data['gpt1'], train_data['gpt2']]),
                           'label':[0]*train_data.shape[0] + [1]*train_data.shape[0]*2})

test_data = pd.DataFrame({'text':pd.concat([test_data['answer'], test_data['gpt2']]),
                           'label':[0]*test_data.shape[0]+[1]*test_data.shape[0]})

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)
train_dataset, test_dataset

## Build model

In [ ]:
batch_size = 6
num_epochs = 3

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train_data = train_dataset.map(preprocess_function, batched=True)
tokenized_test_data = test_dataset.map(preprocess_function, batched=True)

import evaluate
import numpy as np
import tensorflow as tf
from transformers import DataCollatorWithPadding
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers.keras_callbacks import KerasMetricCallback

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return f1.compute(predictions=predictions, references=labels)

batches_per_epoch = len(tokenized_train_data) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)
optimizer, schedule = create_optimizer(init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps)

id2label = {0:'human', 1:'ai'}
label2id = {'human':0, 'ai':1}
model = TFAutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

tf_train_set = model.prepare_tf_dataset(
    tokenized_train_data,
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_validation_set = model.prepare_tf_dataset(
    tokenized_test_data,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

model.compile(optimizer=optimizer)
metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)

# Train the model

We directly use the `test_data` as `validation_data` for RoBERTa and record its F1 at the end.

In [ ]:
model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=1, callbacks=[metric_callback])